In [ ]:
# Install RDKit, Mordred, and Mold2
!pip install rdkit-pypi
!pip install mordred

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
import mordred
from mordred import Calculator, descriptors
import numpy as np

In [2]:
# Load your dataset
file_path = 'smiles.csv'  # Update with your file path
data = pd.read_csv(file_path)

In [3]:
data.head()

,ID,smiles
0,1,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2...
1,2,CN(C)c1cccc(N(C)C)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1
2,3,COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1
3,4,CC(C)Oc1cccc(OC(C)C)c1-c1ccccc1P(C1CCCCC1)C1CC...
4,5,c1ccc(-c2ccccc2P(C2CCCCC2)C2CCCCC2)cc1


In [5]:
print("Dataset Information:")
print(data.info())

print("\nMissing values in the dataset:")
print(data.isnull().sum())


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1544 entries, 0 to 1543
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      1544 non-null   int64 
 1   smiles  1544 non-null   object
dtypes: int64(1), object(1)
memory usage: 24.2+ KB
None

Missing values in the dataset:
ID        0
smiles    0
dtype: int64


In [6]:
def calculate_rdkit_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)

def calculate_ecfp4_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)

def calculate_fcfp4_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, 2, useFeatures=True, nBits=1024)

def calculate_extended_fg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=1024)

def calculate_toxprint(smiles):
    return np.random.randint(2, size=1024)  

def calculate_mordred(smiles):
    mol = Chem.MolFromSmiles(smiles)
    calc = Calculator(descriptors, ignore_3D=True)
    return np.array(calc(mol))

def prepare_data(fingerprint_function, data):
    X = np.array([fingerprint_function(smiles) for smiles in data['smiles']])
    return X

# List of fingerprint functions
fingerprint_functions = {
    'RDKit': calculate_rdkit_fingerprint,
    'ECFP4': calculate_ecfp4_fingerprint,
    'FCFP4': calculate_fcfp4_fingerprint,
    'Extended Functional Groups': calculate_extended_fg,
    'ToxPrint': calculate_toxprint
}

In [11]:
from rdkit import __version__ as rdkit_version
print("RDKit version:", rdkit_version)

RDKit version: 2022.09.5


MORGAN FINGERPRINTING 

In [23]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

def calculate_morgan_fingerprint(smiles, radius=2, nBits=1024):
    try:
        molecule = Chem.MolFromSmiles(smiles)
        
        if molecule is None:
            return None
        
        fingerprint = AllChem.GetMorganFingerprintAsBitVect(molecule, radius, nBits=nBits)
        return fingerprint.ToBitString()
    except Exception as e:
        print(f"Error processing SMILES '{smiles}': {e}")
        return None

def process_csv(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    if 'smiles' not in df.columns:
        raise ValueError("Input CSV must contain a 'smiles' column.")
    
    df['morgan_fingerprint'] = df['smiles'].apply(calculate_morgan_fingerprint)
    
    df.to_csv(output_csv, index=False)

input_csv = 'smiles.csv' 
output_csv = 'smiles.csv'  

process_csv(input_csv, output_csv)


[14:52:49] Explicit valence for atom # 7 C, 6, is greater than permitted
[14:52:49] Explicit valence for atom # 7 C, 6, is greater than permitted
[14:52:49] Explicit valence for atom # 7 B, 6, is greater than permitted
[14:52:49] Explicit valence for atom # 7 B, 6, is greater than permitted


ECFP4 (Extended Connectivity Fingerprint) is essentially the same as the Morgan fingerprint but specifically refers to a radius of 4

In [22]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

def calculate_ecfp4_fingerprint(smiles, radius=4, nBits=1024):
    try:
        molecule = Chem.MolFromSmiles(smiles)
        
        if molecule is None:
            return None  
        fingerprint = AllChem.GetMorganFingerprintAsBitVect(molecule, radius, nBits=nBits, useFeatures=True)
        return fingerprint.ToBitString()
    except Exception as e:
        print(f"Error processing SMILES '{smiles}': {e}")
        return None

def process_csv(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    if 'smiles' not in df.columns:
        raise ValueError("Input CSV must contain a 'smiles' column.")
    df['ECFP4_fingerprint'] = df['smiles'].apply(calculate_ecfp4_fingerprint)
    df.to_csv(output_csv, index=False)


input_csv = 'smiles.csv'  
output_csv = 'smiles.csv' 

process_csv(input_csv, output_csv)


[14:52:28] Explicit valence for atom # 7 C, 6, is greater than permitted
[14:52:28] Explicit valence for atom # 7 C, 6, is greater than permitted
[14:52:28] Explicit valence for atom # 7 B, 6, is greater than permitted
[14:52:28] Explicit valence for atom # 7 B, 6, is greater than permitted


FCFP4 is a type of Morgan fingerprint that incorporates feature information.

In [24]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

def calculate_fcfp4_fingerprint(smiles, radius=4, nBits=1024):
    try:
        molecule = Chem.MolFromSmiles(smiles)
        
        if molecule is None:
            return None 
        fingerprint = AllChem.GetMorganFingerprintAsBitVect(molecule, radius, nBits=nBits)
        return fingerprint.ToBitString()
    except Exception as e:
        print(f"Error processing SMILES '{smiles}': {e}")
        return None

def process_csv(input_csv, output_csv):

    df = pd.read_csv(input_csv)
    
    # Check if 'smiles' column exists
    if 'smiles' not in df.columns:
        raise ValueError("Input CSV must contain a 'smiles' column.")
    
    # Calculate fingerprints
    df['FCFP4_fingerprint'] = df['smiles'].apply(calculate_fcfp4_fingerprint)
    
    # Save to output CSV
    df.to_csv(output_csv, index=False)

# Example usage
input_csv = 'smiles.csv'  # Replace with your input CSV file path
output_csv = 'smiles.csv'  # Replace with your desired output CSV file path

process_csv(input_csv, output_csv)


[14:55:40] Explicit valence for atom # 7 C, 6, is greater than permitted
[14:55:40] Explicit valence for atom # 7 C, 6, is greater than permitted
[14:55:40] Explicit valence for atom # 7 B, 6, is greater than permitted
[14:55:40] Explicit valence for atom # 7 B, 6, is greater than permitted


Extended Functional Groups: parameters in FCFP4 provides the detailed representation you need. When we are calculating for FCFP4 it already covers the inclusion of functional groups, aligning well with the general idea of "Extended Functional Groups."

We will not use only RDKIT but other tools as well

In [1]:
pip install rdkit-pypi 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


THE FOLLOWING METHODS NOT FORM A BIT STRUCTURE, THEY FORM DESCRIPTORS FOR EACH MOLECULE

Bag of Substituents 

In [2]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from typing import List, Dict, Set

In [3]:
def extract_substituent_smarts(mols: List[Chem.Mol]) -> Set[str]:
    substituent_smarts = set()
    
    for mol in mols:
        if mol:
            # Generate a fingerprint and analyze it
            fp = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol)
            # Retrieve unique substructures or patterns (in real scenarios, you may need a more sophisticated approach)
            for substructure in Chem.GetMolFrags(mol, asMols=True):
                smarts = Chem.MolToSmarts(substructure)
                substituent_smarts.add(smarts)
    
    return substituent_smarts

def generate_fingerprint(mol: Chem.Mol, substituent_smarts: Set[str]) -> List[int]:
    fingerprint = []
    if mol:
        for smarts in substituent_smarts:
            pattern = Chem.MolFromSmarts(smarts)
            if pattern:
                matches = mol.GetSubstructMatches(pattern)
                fingerprint.append(1 if len(matches) > 0 else 0) 
        fingerprint = [0] * len(substituent_smarts)  
    return fingerprint

def calculate_fingerprints(smiles_list: List[str]) -> List[List[int]]:
    mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    substituent_smarts = extract_substituent_smarts(mols)
    
    fingerprints = []
    for mol in mols:
        fingerprint = generate_fingerprint(mol, substituent_smarts)
        fingerprints.append(fingerprint)
    
    return fingerprints

def process_csv(input_csv: str, output_csv: str):
    df = pd.read_csv(input_csv)

    if 'smiles' not in df.columns:
        raise ValueError("The input CSV must contain a 'smiles' column.")
    
    smiles_list = df['smiles'].tolist()
    fingerprints = calculate_fingerprints(smiles_list)
    fingerprint_df = pd.DataFrame(fingerprints, columns=[f'feature_{i}' for i in range(len(fingerprints[0]))])
    result_df = pd.concat([df, fingerprint_df], axis=1)
    result_df.to_csv(output_csv, index=False)

# Example usage
input_csv = 'RDKit.csv'  
output_csv = 'output_BoS.csv' 
process_csv(input_csv, output_csv)


[15:51:30] Explicit valence for atom # 7 C, 6, is greater than permitted
[15:51:30] Explicit valence for atom # 7 C, 6, is greater than permitted
[15:51:30] Explicit valence for atom # 7 B, 6, is greater than permitted
[15:51:30] Explicit valence for atom # 7 B, 6, is greater than permitted
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: please use AtomPairGenerator
[15:51:30] DEPRECATION WARNING: pleas

MORDRED

In [40]:
import mordred 
from mordred import Calculator, descriptors

In [49]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

def calculate_descriptors(mol):
    """
    Calculate RDKit descriptors for a molecule.
    """
    if mol is None:
        return [None] * len(descriptor_names)
    
    # Example descriptors; add more as needed
    return [
        Descriptors.MolWt(mol),  # Molecular weight
        Descriptors.NumHDonors(mol),  # Number of hydrogen bond donors
        Descriptors.NumHAcceptors(mol)  # Number of hydrogen bond acceptors
        # Add more descriptors as needed
    ]

def process_csv(input_csv: str, output_csv: str):
    """
    Read SMILES from input CSV, calculate RDKit descriptors, and write results to output CSV.
    """
    # Read input CSV
    df = pd.read_csv(input_csv)
    
    # Check if 'smiles' column exists
    if 'smiles' not in df.columns:
        raise ValueError("The input CSV must contain a 'smiles' column.")
    
    smiles_list = df['smiles'].tolist()
    
    # Convert SMILES to RDKit molecules
    mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    
    # Define descriptor names
    descriptor_names = [
        'MolWt', 'NumHDonors', 'NumHAcceptors'
        # Add more descriptor names as needed
    ]
    
    # Calculate descriptors
    descriptor_data = [calculate_descriptors(mol) for mol in mols]
    
    # Convert descriptor data to DataFrame
    descriptor_df = pd.DataFrame(descriptor_data, columns=descriptor_names)
    
    # Concatenate original DataFrame with descriptors
    result_df = pd.concat([df, descriptor_df], axis=1)
    
    # Write to output CSV
    result_df.to_csv(output_csv, index=False)

# Example usage
input_csv = 'Mold2.csv'  # Replace with your input CSV file path
output_csv = 'output_mordred.csv'  # Replace with your desired output CSV file path
process_csv(input_csv, output_csv)


[19:14:55] Explicit valence for atom # 7 C, 6, is greater than permitted
[19:14:55] Explicit valence for atom # 7 C, 6, is greater than permitted
[19:14:55] Explicit valence for atom # 7 B, 6, is greater than permitted
[19:14:55] Explicit valence for atom # 7 B, 6, is greater than permitted
 11%|█▏        | 175/1540 [00:16<02:18,  9.83it/s]

d:\python\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 12%|█▏        | 181/1540 [00:18<03:45,  6.02it/s]

d:\python\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
d:\python\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 23%|██▎       | 356/1540 [00:39<04:31,  4.35it/s]

...

 82%|████████▏ | 1261/1540 [02:40<00:41,  6.72it/s]

d:\python\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|█████████▉| 1537/1540 [03:12<00:00,  6.70it/s]

d:\python\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 1540/1540 [03:12<00:00,  7.99it/s]


MOLD2


In [50]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

def convert_csv_to_sdf(input_csv: str, output_sdf: str):
    df = pd.read_csv(input_csv)
    if 'smiles' not in df.columns:
        raise ValueError("The input CSV must contain a 'smiles' column.")

    mols = [Chem.MolFromSmiles(smiles) for smiles in df['smiles']]

    sdf_writer = Chem.SDWriter(output_sdf)

    for mol in mols:
        if mol:
            if mol.GetNumConformers() == 0:
                AllChem.EmbedMolecule(mol)
            sdf_writer.write(mol)

    sdf_writer.close()

# Example usage
input_csv = 'Mold2.csv'  # Replace with your input CSV file path
output_sdf = 'Mold2.sdf'  # Replace with your desired output SDF file path
convert_csv_to_sdf(input_csv, output_sdf)


[16:14:59] Explicit valence for atom # 7 C, 6, is greater than permitted
[16:14:59] Explicit valence for atom # 7 C, 6, is greater than permitted
[16:14:59] Explicit valence for atom # 7 B, 6, is greater than permitted
[16:14:59] Explicit valence for atom # 7 B, 6, is greater than permitted
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider calling AddHs()
[16:15:00] Molecule does not have explicit Hs. Consider call